In [2]:
from joblib import Parallel, delayed
import os
import hydrobr
import time
import pandas as pd

In [3]:
df_station_list = hydrobr.get_data.ANA.list_prec_stations()
df_station_list

,Name,Code,Type,SubBasin,City,State,Responsible,Latitude,Longitude,StartDate,EndDate,NYD,MD,N_YWOMD,YWMD
0,SALINÓPOLIS,00047000,2,32,SALINÓPOLIS,PARÁ,INMET,-0.6500,-47.5500,1958/01/01,1964/12/31,7,25.0,0,100.0
1,SALINÓPOLIS,00047002,2,32,SALINÓPOLIS,PARÁ,ANA,-0.6231,-47.3536,1977/12/09,2019/08/31,43,3.5,35,18.6
2,CURUÇA,00047003,2,32,CURUÇA,PARÁ,ANA,-0.7375,-47.8536,1981/07/01,2019/07/31,39,2.4,29,25.6
3,PRIMAVERA,00047004,2,32,PRIMAVERA,PARÁ,ANA,-0.9294,-47.0994,1982/02/18,2019/08/31,38,0.0,35,7.9
4,MARUDA,00047005,2,32,MARAPANIM,PARÁ,ANA,-0.6336,-47.6583,1989/08/21,2019/07/31,31,5.0,20,35.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11971,NOVA ESPERANÇA/MARCO BV-8,08461000,2,14,PACARAIMA,RORAIMA,ANA,4.4883,-61.1297,1984/03/23,2019/06/30,36,12.3,13,63.9
11972,MISSÃO AUARIS - JUSANTE,08464001,2,14,BOA VISTA,RORAIMA,ANA,4.0031,-64.4431,1995/04/01,2019/06/30,25,4.2,15,40.0
11973,WILLIAM KRAANPLEIN,08555060,2,90,SURINAME,SURINAME,SURINAME,5.8000,-55.1667,1935/08/01,1948/12/31,14,9.2,7,50.0
11974,ZANDERIJ,08555096,2,90,SURINAME,SURINAME,SURINAME,5.4700,-55.2000,2009/01/31,2010/01/31,2,96.7,0,100.0


In [4]:
df_station_list_br = df_station_list[~df_station_list['State'].isin(['BOLÍVIA', 'PERU', 'PARAGUAI', 'ARGENTINA', 'GUIANA FRANCESA', 'SURINAME'])]
len(df_station_list_br['State'].unique().tolist())

27

In [5]:
gauge_codes_total = df_station_list_br['Code'].unique().tolist()
len(gauge_codes_total)

11893

In [7]:
filenames = [f for f in os.listdir(os.path.join('./1 - Raw Data/')) if f.endswith('.h5')]
codes = [os.path.splitext(f)[0] for f in filenames]
print(codes[0], len(codes))

00047000 11782


In [29]:
gauge_codes = [code for code in gauge_codes_total if code not in codes]
print(len(gauge_codes), gauge_codes[:10])

111 ['00635130', '00635131', '00635132', '00635133', '00635134', '00635135', '00637072', '00637073', '00637074', '00637075']


In [30]:
def fetch_and_save(gauge_code):
    try:
        df = hydrobr.get_data.ANA.prec_data([gauge_code])
        out_dir = os.path.join('./1 - Raw Data/')
        os.makedirs(out_dir, exist_ok=True)
        out_path = os.path.join(out_dir, f'{gauge_code}.h5')
        df.to_hdf(out_path, key='table_data', mode='w', format='table',
                  complevel=9, complib='zlib', encoding='utf-8')
        return (gauge_code, True, None)
    except Exception as e:
        return (gauge_code, False, str(e))

In [31]:
def error_listing(results):
    errors = [(r[0], r[2]) for r in results if not r[1]]
    df_errors = pd.DataFrame(errors, columns=['gauge_code', 'error_message'])
    df_errors.to_excel('download_errors.xlsx', index=False)
    df_success = pd.DataFrame([r[0] for r in results if r[1]], columns=['gauge_code'])
    df_success.to_excel('download_success.xlsx', index=False)
    gauge_codes_error_list = df_errors['gauge_code'].unique().tolist()
    return gauge_codes_error_list

In [32]:
def get_station_data(gauge_codes):
    results = Parallel(n_jobs=-1, prefer="threads")(
        delayed(fetch_and_save)(gc) for gc in gauge_codes
    )
    success = [r[0] for r in results if r[1]]
    errors = [(r[0], r[2]) for r in results if not r[1]]
    print(f"Total success: {len(success)}, total errors: {len(errors)}")
    return results, success, errors

In [33]:
results = []


In [34]:
new_error_count = None
old_error_count = None

In [ ]:
while new_error_count != 0:
    if results:
        gauge_codes = error_listing(results)
        if old_error_count == new_error_count or new_error_count == 0:
            print('No more errors to retry...')
            # answer = input("Do you want to exit? (y/n): ")
            # if answer.lower() == 'y':
            #     exit()
        old_error_count = int(new_error_count)
        print(f'Retrying {len(gauge_codes)} failed downloads...')
        results, success, errors = get_station_data(gauge_codes)
        new_error_count = len(errors)
    else:
        print(f'Downloading {len(gauge_codes)} gauge codes...')
        results, success, errors = get_station_data(gauge_codes)
        new_error_count = len(errors)

  0%|          | 0/1 [00:00<?, ?it/s]








































































































  0%|          | 0/1 [00:00<?, ?it/s]










  0%|          | 0/1 [00:00<?, ?it/s]









  0%|          | 0/1 [00:00<?, ?it/s]




























































100%|██████████| 1/1 [00:00<00:00,  2.93it/s]

  0%|          | 0/1 [00:00<?, ?it/s]







100%|██████████| 1/1 [00:00<00:00,  2.78it/s]








  0%|          | 0/1 [00:00<?, ?it/s]










  0%|          | 0/1 [00:00<?, ?it/s]



  0%|          | 0/1 [00:00<?, ?it/s]

















  0%|          | 0/1 [00:00<?, ?it/s]

























  0%|          | 0/1 [00:00<?, ?it/s]














  0%|          | 0/1 [00:00<?, ?it/s][A








  0%|          | 0/1 [00:00<?, ?it/s]










  0%|          | 0/1 [00:00<?, ?it/s]



  0%|          | 0/1 [00:00<?, ?it/s]

















  0%|          | 0/1 [00:00<?, ?it/s]

























  0%|

Total success: 0, total errors: 111
Retrying 111 failed downloads...


  0%|          | 0/1 [00:00<?, ?it/s]








































































































  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  5.61it/s]

100%|██████████| 1/1 [00:00<00:00,  5.52it/s]



  0%|          | 0/1 [00:00<?, ?it/s]
















  0%|          | 0/1 [00:00<?, ?it/s]











  0%|          | 0/1 [00:00<?, ?it/s]
















































  0%|          | 0/1 [00:00<?, ?it/s]



  0%|          | 0/1 [00:00<?, ?it/s]






  0%|          | 0/1 [00:00<?, ?it/s]














  0%|          | 0/1 [00:00<?, ?it/s]
















  0%|          | 0/1 [00:00<?, ?it/s]









  0%|          | 0/1 [00:00<?, ?it/s]













  0%|          | 0/1 [00:00<?, ?it/s][A













  0%|          | 0/1 [00:00<?, ?it/s][A

  0%|          | 0/1 [00:00<?, ?it/s]


  0%|          | 0/1 [00:00<?, ?it/s]






  0%|          | 0/1 [00:00<?, ?it/s]










  0%|          | 0/

Total success: 0, total errors: 111
No more errors to retry...
Retrying 111 failed downloads...


  0%|          | 0/1 [00:00<?, ?it/s]








































































































100%|██████████| 1/1 [00:00<00:00,  5.50it/s]




100%|██████████| 1/1 [00:00<00:00,  5.44it/s]




  0%|          | 0/1 [00:00<?, ?it/s]









  0%|          | 0/1 [00:00<?, ?it/s]






  0%|          | 0/1 [00:00<?, ?it/s]











  0%|          | 0/1 [00:00<?, ?it/s]











  0%|          | 0/1 [00:00<?, ?it/s]






































  0%|          | 0/1 [00:00<?, ?it/s]




  0%|          | 0/1 [00:00<?, ?it/s]









  0%|          | 0/1 [00:00<?, ?it/s]





  0%|          | 0/1 [00:00<?, ?it/s]











  0%|          | 0/1 [00:00<?, ?it/s]




































  0%|          | 0/1 [00:00<?, ?it/s]


















  0%|          | 0/1 [00:00<?, ?it/s]









  0%|          | 0/1 [00:00<?, ?it/s]





  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]










  0%|

...

  0%|          | 0/1 [00:00<?, ?it/s]








































































































  0%|          | 0/1 [00:00<?, ?it/s]







  0%|          | 0/1 [00:00<?, ?it/s]









  0%|          | 0/1 [00:00<?, ?it/s]








  0%|          | 0/1 [00:00<?, ?it/s]






















  0%|          | 0/1 [00:00<?, ?it/s]
















100%|██████████| 1/1 [00:00<00:00,  4.54it/s]















  0%|          | 0/1 [00:00<?, ?it/s][A











  0%|          | 0/1 [00:00<?, ?it/s]









  0%|          | 0/1 [00:00<?, ?it/s]








  0%|          | 0/1 [00:00<?, ?it/s]





























  0%|          | 0/1 [00:00<?, ?it/s]














  0%|          | 0/1 [00:00<?, ?it/s][A


  0%|          | 0/1 [00:00<?, ?it/s]













  0%|          | 0/1 [00:00<?, ?it/s][A











  0%|          | 0/1 [00:00<?, ?it/s]









  0%|          | 0/1 [00:00<?, ?it/s]








  0%|          | 0/1 [00:00<?, ?it/s]









Total success: 0, total errors: 111
No more errors to retry...
Retrying 111 failed downloads...


  0%|          | 0/1 [00:00<?, ?it/s]








































































































100%|██████████| 1/1 [00:00<00:00,  6.20it/s]

  0%|          | 0/1 [00:00<?, ?it/s]





  0%|          | 0/1 [00:00<?, ?it/s]





























  0%|          | 0/1 [00:00<?, ?it/s]











  0%|          | 0/1 [00:00<?, ?it/s]


































  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]





  0%|          | 0/1 [00:00<?, ?it/s]













  0%|          | 0/1 [00:00<?, ?it/s]
















  0%|          | 0/1 [00:00<?, ?it/s]













































  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]





  0%|          | 0/1 [00:00<?, ?it/s]













  0%|          | 0/1 [00:00<?, ?it/s][A







  0%|          | 0/1 [00:00<?, ?it/s]









  0%|          | 0/1 [00:00<?, ?it/s]














  0%|          | 0/1 [00:00<?, 

Total success: 0, total errors: 111
No more errors to retry...
Retrying 111 failed downloads...


  0%|          | 0/1 [00:00<?, ?it/s]








































































































100%|██████████| 1/1 [00:00<00:00,  6.11it/s]

100%|██████████| 1/1 [00:00<00:00,  4.68it/s]


  0%|          | 0/1 [00:00<?, ?it/s]










  0%|          | 0/1 [00:00<?, ?it/s]



















  0%|          | 0/1 [00:00<?, ?it/s]


















































  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]


  0%|          | 0/1 [00:00<?, ?it/s]








  0%|          | 0/1 [00:00<?, ?it/s]



















  0%|          | 0/1 [00:00<?, ?it/s]




































  0%|          | 0/1 [00:00<?, ?it/s][A














  0%|          | 0/1 [00:00<?, ?it/s][A

  0%|          | 0/1 [00:00<?, ?it/s]


  0%|          | 0/1 [00:00<?, ?it/s]








  0%|          | 0/1 [00:00<?, ?it/s]



















  0%|          | 0/1 [00:00<?, ?it/s]











  0%|          | 0/1 [00:00